# 01 · Geração do dataset ToLD-Br e arquivos para o IAA

- **Origem:** `experiments/generate_dataset.ipynb`
- **Faz:** agrega os votos dos 3 anotadores em `ToLD-BR.csv` (6 categorias) e gera os TSVs por categoria para o cálculo de concordância entre anotadores (IAA); fixa a Tabela 6 do artigo como asserção reproduzível (21.000 / 9.255 / 11.745).
- **Entradas:** `ToLD-BR_alpha.csv` — anotações brutas (variável `ALPHA_CSV`).
- **Saídas:** `results/ToLD-BR_generated.csv` (idêntico ao `ToLD-BR.csv` do repo) e `results/{categoria}_alpha.tsv`.

> **Diff mínimo.** Cada alteração abaixo é marcada **[T]** tecnologia descontinuada, **[B]** bug ou **[A]** fidelidade ao artigo — catálogo completo em [`CHANGES.md`](CHANGES.md).

In [6]:
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "ToLD-BR.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_ZIP  = ROOT / "experiments" / "data" / "1annotator.zip"
MAIN_CSV  = ROOT / "ToLD-BR.csv"
ALPHA_CSV = ROOT / "ToLD-BR_alpha.csv"
RESULTS   = ROOT / "reproduction" / "results"
FIGURES   = RESULTS / "figures"
RESULTS.mkdir(parents=True, exist_ok=True); FIGURES.mkdir(parents=True, exist_ok=True)

In [7]:
import pandas as pd
import numpy as np

**Mudança #1 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | caminho de entrada do CSV de anotações |
| **Antes** | `pd.read_csv("../data/told-br/ToLD-BR_alpha.csv")` (caminho relativo da estrutura antiga `data/told-br/`) |
| **Depois** | `pd.read_csv(ALPHA_CSV)` — lê `ToLD-BR_alpha.csv` da raiz do repo |
| **Motivo** | o notebook agora roda de `reproduction/notebooks/` e a estrutura de dados foi achatada; o arquivo de anotações vive na raiz do repo como `ToLD-BR_alpha.csv` (variável `ALPHA_CSV`) |
| **Impacto** | nenhum no conteúdo — mesma leitura, apenas o caminho muda |

In [8]:
data = pd.read_csv(ALPHA_CSV)

In [9]:
data["toxic"] = 0
data["toxic_1"] = 0
data["toxic_2"] = 0
data["toxic_3"] = 0
data["homophobia"] = 0
data["obscene"] = 0
data["insult"] = 0
data["racism"] = 0
data["misogyny"] = 0
data["xenophobia"] = 0
for i, row in data.iterrows():
    count = 0
    if 1 in list(row[["homophobia_1", "obscene_1", "insult_1", "racism_1", "misogyny_1", "xenophobia_1"]]):
        data.loc[i, "toxic_1"] = 1
        count += 1
    if 1 in list(row[["homophobia_2", "obscene_2", "insult_2", "racism_2", "misogyny_2", "xenophobia_2"]]):
        data.loc[i, "toxic_2"] = 1
        count += 1
    if 1 in list(row[["homophobia_3", "obscene_3", "insult_3", "racism_3", "misogyny_3", "xenophobia_3"]]):
        data.loc[i, "toxic_3"] = 1
        count += 1

    data.loc[i, "homophobia"] = np.sum(list(row[["homophobia_1", "homophobia_2", "homophobia_3"]]))
    
    data.loc[i, "obscene"] = np.sum(list(row[["obscene_1", "obscene_2", "obscene_3"]]))
    
    data.loc[i, "insult"] = np.sum(list(row[["insult_1", "insult_2", "insult_3"]]))
    
    data.loc[i, "racism"] = np.sum(list(row[["racism_1", "racism_2", "racism_3"]]))
        
    data.loc[i, "misogyny"] = np.sum(list(row[["misogyny_1", "misogyny_2", "misogyny_3"]]))

    data.loc[i, "xenophobia"] = np.sum(list(row[["xenophobia_1", "xenophobia_2", "xenophobia_3"]]))
        
    if count >= 1:
        data.loc[i, "toxic"] = 1

In [10]:
data[["text", "toxic", "homophobia", "obscene", "insult", "racism", "misogyny", "xenophobia"]]

,text,toxic,homophobia,obscene,insult,racism,misogyny,xenophobia
0,Meu nivel de amizade com isis é ela ter meu in...,1,0,0,2,0,0,0
1,"rt @user @user o cara adultera dados, que fora...",1,0,0,1,0,0,0
2,@user @user @user o cara só é simplesmente o m...,1,0,2,1,0,0,0
3,eu to chorando vei vsf e eu nem staneio izone ...,1,0,1,0,0,0,0
4,Eleitor do Bolsonaro é tão ignorante q não per...,1,0,1,2,0,0,0
...,...,...,...,...,...,...,...,...
20995,@user faz favor vai carai,0,0,0,0,0,0,0
20996,só queria conhecer alguém que não conhece o he...,1,1,0,0,0,0,0
20997,"vcs militam na hora errada em cima de memes, p...",0,0,0,0,0,0,0
20998,@user porra any eu tava c dor de cabeca e fui ...,0,0,0,0,0,0,0


**Mudança #2 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | destino do CSV gerado + validação contra o repo |
| **Antes** | `data[[...]].to_csv("../data/told-br/ToLD-BR.csv", index=False)` (sobrescrevia o arquivo versionado em estrutura de pastas antiga) |
| **Depois** | grava em `RESULTS/ToLD-BR_generated.csv` e valida igualdade (`assert_frame_equal`) com o `ToLD-BR.csv` versionado (`MAIN_CSV`) |
| **Motivo** | não sobrescrever silenciosamente o `ToLD-BR.csv` versionado na raiz; gravamos em `RESULTS/ToLD-BR_generated.csv` e validamos que o conteúdo gerado é igual ao `ToLD-BR.csv` do repositório (variável `MAIN_CSV`) |
| **Impacto** | nenhum na geração; adiciona prova de reprodução (saída gerada == artefato versionado) |

In [11]:
# MUDANCA #2 (T): grava em RESULTS (sem sobrescrever o repo) e valida igualdade com ToLD-BR.csv versionado
cols_main = ["text", "homophobia", "obscene", "insult", "racism", "misogyny", "xenophobia"]
generated_csv = RESULTS / "ToLD-BR_generated.csv"
data[cols_main].to_csv(generated_csv, index=False)
print(f"CSV gerado em: {generated_csv}")

# Validacao: o conteudo gerado deve ser identico ao ToLD-BR.csv versionado na raiz do repo.
# (As colunas de categoria sao float64 em ambos, pois a agregacao usa np.sum; check_dtype=False e defensivo.)
gen = pd.read_csv(generated_csv)
repo = pd.read_csv(MAIN_CSV)
assert list(gen.columns) == list(repo.columns) == cols_main, (
    f"Colunas divergentes.\n  geradas: {list(gen.columns)}\n  repo:    {list(repo.columns)}"
)
assert gen.shape == repo.shape, f"Shapes divergentes: gerado {gen.shape} vs repo {repo.shape}"
pd.testing.assert_frame_equal(
    gen.reset_index(drop=True), repo.reset_index(drop=True), check_dtype=False
)
print(f"OK: {generated_csv.name} e identico ao {MAIN_CSV.name} versionado (shape {gen.shape}).")

CSV gerado em: C:\Users\Pedro\OneDrive\Documentos\UFCG\Mestrado\2026.1\Projeto - Reprodução\ToLD-Br\reproduction\results\ToLD-BR_generated.csv
OK: ToLD-BR_generated.csv e identico ao ToLD-BR.csv versionado (shape (21000, 7)).


**Mudança #4 · [A] fidelidade**

| | |
|:--|:--|
| **O quê** | checagem de sanidade contra a Tabela 6 do artigo |
| **Antes** | `data[["text", "toxic"]]["toxic"].value_counts()` (apenas exibia as contagens) |
| **Depois** | `assert` de 21.000 linhas / 9.255 tóxicos / 11.745 não-tóxicos |
| **Motivo** | fidelidade ao artigo — fixar como teste reproduzível os números publicados (21.000 linhas; 9.255 tóxicos / 11.745 não-tóxicos) com `assert`, em vez de inspeção visual |
| **Impacto** | nenhum no dataset; adiciona verificação automática que falha se a agregação divergir do artigo |

In [12]:
# Sanidade contra a Tabela 6 do artigo (2010.04543v1)
counts = data["toxic"].value_counts()
n_total = len(data)
n_toxic = int(counts.get(1, 0))
n_nontoxic = int(counts.get(0, 0))
print(f"Total de linhas: {n_total}")
print(f"Toxicos (toxic=1): {n_toxic}")
print(f"Nao-toxicos (toxic=0): {n_nontoxic}")

assert n_total == 21000, f"Esperado 21000 linhas, obtido {n_total}"
assert n_toxic == 9255, f"Esperado 9255 toxicos, obtido {n_toxic}"
assert n_nontoxic == 11745, f"Esperado 11745 nao-toxicos, obtido {n_nontoxic}"
print("OK: contagens batem com a Tabela 6 do artigo.")

Total de linhas: 21000
Toxicos (toxic=1): 9255
Nao-toxicos (toxic=0): 11745
OK: contagens batem com a Tabela 6 do artigo.


**Mudança #3 · [T] tecnologia**

| | |
|:--|:--|
| **O quê** | destino dos TSVs por categoria para o IAA |
| **Antes** | `final_df.fillna("?").to_csv(f"{category}_alpha.tsv", ...)` (gravava no diretório de trabalho atual) |
| **Depois** | `final_df.fillna("?").to_csv(RESULTS / f"{category}_alpha.tsv", ...)` — saída centralizada em `results/` |
| **Motivo** | o notebook roda de `reproduction/notebooks/`; centralizamos os artefatos em `RESULTS` para não poluir a pasta de notebooks e para que o notebook de IAA consiga encontrá-los |
| **Impacto** | nenhum no conteúdo dos TSVs; apenas o diretório de saída muda |

In [13]:
# Generate files for IAA using mwetoolkit3 (TSVs por categoria)
# MUDANCA #3 (T): saida em RESULTS em vez do cwd.
for category in ["homophobia", "obscene", "insult", "racism", "misogyny", "xenophobia"]:
    final_df = pd.DataFrame(columns=range(0, 42), index=range(21000))
    j = 0
    for i in range(0, 14):
        df = data[[f"{category}_1", f"{category}_2", f"{category}_3"]].iloc[i * 1500:1500 * (i + 1)]
        for col in df:
            final_df.iloc[i * 1500:1500 * (i + 1), j] = df[col]
            j += 1

    out_tsv = RESULTS / f"{category}_alpha.tsv"
    final_df.fillna("?").to_csv(out_tsv, index=False, sep="\t")
    print(f"TSV gerado: {out_tsv}")

TSV gerado: C:\Users\Pedro\OneDrive\Documentos\UFCG\Mestrado\2026.1\Projeto - Reprodução\ToLD-Br\reproduction\results\homophobia_alpha.tsv
TSV gerado: C:\Users\Pedro\OneDrive\Documentos\UFCG\Mestrado\2026.1\Projeto - Reprodução\ToLD-Br\reproduction\results\obscene_alpha.tsv
TSV gerado: C:\Users\Pedro\OneDrive\Documentos\UFCG\Mestrado\2026.1\Projeto - Reprodução\ToLD-Br\reproduction\results\insult_alpha.tsv
TSV gerado: C:\Users\Pedro\OneDrive\Documentos\UFCG\Mestrado\2026.1\Projeto - Reprodução\ToLD-Br\reproduction\results\racism_alpha.tsv
TSV gerado: C:\Users\Pedro\OneDrive\Documentos\UFCG\Mestrado\2026.1\Projeto - Reprodução\ToLD-Br\reproduction\results\misogyny_alpha.tsv
TSV gerado: C:\Users\Pedro\OneDrive\Documentos\UFCG\Mestrado\2026.1\Projeto - Reprodução\ToLD-Br\reproduction\results\xenophobia_alpha.tsv


## Calculo do IAA (kappa)
Os TSVs por categoria foram gravados em `reproduction/results/`. Para cada CATEGORY, o calculo de concordancia entre anotadores e feito com mwetoolkit3:
`python mwetoolkit3/bin/kappa.py -i -r -c reproduction/results/CATEGORY_alpha.tsv`
(Esse passo e consumido pelo notebook de IAA correspondente.)